<a href="https://colab.research.google.com/github/gcbenito1-blip/elective_streamlit/blob/main/scraper_cleaner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install translate
!pip install google-play-scraper
!pip install langdetect
!pip install translate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 19.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=a581307732cfd5bad47fb2c8ed6b1103691a3a3af028066f1f366d15edda16fc
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [ ]:
from google_play_scraper import Sort, reviews_all
from langdetect import detect
import pandas as pd
import re
import unicodedata

In [ ]:
def scrape_data():

    results = reviews_all(
        'egov.app',
        lang="en",
        country="ph",
        sort=Sort.NEWEST,
    )

    return results

def detect_lang(row):
    try:
        if detect(row['content']) == 'tl':
            return 1
        else:
            return 0
    except:
        return 0

In [ ]:
review_dataset = scrape_data()
df = pd.DataFrame(review_dataset)

df = df.reindex(columns=[
    'reviewId', 'content', 'score', 'thumbsUpCount',
    'reviewCreatedVersion', 'at'
])

df.index.name = 'index'
df['content'] = df['content'].astype(str).str.lower()
df['is_tagalog'] = df.apply(detect_lang, axis=1)
df.to_csv('all_reviews.csv')

In [ ]:
def clean_text(text):
    text = unicodedata.normalize('NFKC', str(text))#unicode formatting
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)#http
    text = re.sub(r'@\w+|#\w+', '', text) #removed mentions/hashtags
    text = re.sub(r'<.*?>', '', text) #removed html
    text = re.sub(r'[^\w\s\.!?@#]', '', text) #symbols
    text = re.sub(r'\s+', ' ', text) #space normalization
    return text.lower().strip() #lowerspace and whitespace trimming

emoji_pattern = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F700-\U0001F77F"
    "\U0001F780-\U0001F7FF"
    "\U0001F800-\U0001F8FF"
    "\U0001F900-\U0001F9FF"
    "\U0001FA00-\U0001FAFF"
    "\U00002700-\U000027BF"
    "\U00002600-\U000026FF"
    "]+",
    flags=re.UNICODE
)

def remove_emoji(text):
    return emoji_pattern.sub('', str(text))

df = df.copy()

df['content'] = (
    df['content']
    .astype(str)
    .apply(remove_emoji)
    .apply(clean_text)
)

# Remove empty rows AFTER cleaning
df = df[df['content'].notna() & (df['content'].str.strip() != '')]

df.to_csv('clean_dataset.csv')

In [ ]:
df1 = pd.read_csv('clean_dataset.csv')
df_eng = df1[df1['is_tagalog'] ==0].copy()
df_eng['translated'] = df_eng['content']
df_eng.to_csv('eng_review.csv')
df_tgl = df1[df1['is_tagalog'] ==1].copy()
df_tgl.to_csv('tgl_review.csv')